<a href="https://colab.research.google.com/github/Kavishka2401/CustomerChurnPredictionSystem/blob/master/nn_featureEng_SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount the google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Provide path
data=pd.read_csv('/content/drive/MyDrive/processed_data_NN.csv')

In [ ]:
data_model_3 = data.copy()

In [ ]:
# 1. Average monthly charge per tenure
# Adding 1 to tenure to avoid division by zero
data_model_3['AvgChargePerMonth'] = data_model_3['TotalCharges_numeric'] / (data_model_2['tenure'] + 1)

# 2. Service usage count
# List of columns representing optional services (already one-hot encoded as 0/1)
service_cols = [
    'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
    'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes'
]

# Sum the services used by each customer
data_model_3['ServiceCount'] = data_model_3[service_cols].sum(axis=1)

# Check first few rows to confirm
data_model_3[['AvgChargePerMonth', 'ServiceCount']].head()

In [ ]:
data_model_3.columns

In [ ]:
data_model_3['AvgChargePerMonth'] = data_model_3['TotalCharges_numeric'] / (data_model_2['tenure'] + 1)

service_yes_cols = [
    'PhoneService_Yes',
    'MultipleLines_Yes',
    'InternetService_Fiber optic',  # real service
    'OnlineSecurity_Yes',
    'OnlineBackup_Yes',
    'DeviceProtection_Yes',
    'TechSupport_Yes',
    'StreamingTV_Yes',
    'StreamingMovies_Yes'
]
data_model_3['ServiceCount'] = data_model_3[service_yes_cols].sum(axis=1)

# Check first few rows to confirm
data_model_3[['AvgChargePerMonth', 'ServiceCount']].head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Columns to scale
cols_to_scale = ['AvgChargePerMonth', 'ServiceCount']

# Fit + transform
data_model_3[cols_to_scale] = scaler.fit_transform(data_model_3[cols_to_scale])

In [ ]:
drop_cols = [
    'customerID',
    'MonthlyCharges',
    'TotalCharges',
    'TotalCharges_numeric',
    # ServiceCount component cols
    'PhoneService_Yes',
    'MultipleLines_Yes',
    'InternetService_Fiber optic',
    'OnlineSecurity_Yes',
    'OnlineBackup_Yes',
    'DeviceProtection_Yes',
    'TechSupport_Yes',
    'StreamingTV_Yes',
    'StreamingMovies_Yes'
]

X = data_model_3.drop(columns=drop_cols + ['Churn'])
y = data_model_3['Churn']

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% test
    stratify=y,          # preserve class ratio
    random_state=42
)

# Check shapes and class distribution
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Train class counts:\n", y_train.value_counts())
print("Test class counts:\n", y_test.value_counts())

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only on training data
sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

print("After SMOTE class distribution:")
print(y_train_smote.value_counts())

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

input_dim = X_train_smote.shape[1]

nn_model_smote = Sequential([
    Dense(32, activation='relu', input_dim=input_dim),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

nn_model_smote.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()]
)

In [ ]:
history_smote = nn_model_smote.fit(
    X_train_smote, y_train_smote,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate model
test_metrics = nn_model_smote.evaluate(X_test, y_test, verbose=1)

# Extract metric names
metric_names = nn_model_smote.metrics_names

print("\n Test Set Performance:")
for name, value in zip(metric_names, test_metrics):
    print(f"{name}: {value:.4f}")

In [ ]:
from sklearn.metrics import f1_score

y_pred_prob = nn_model_smote.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

f1 = f1_score(y_test, y_pred)
print("F1-score (0.5 threshold):", f1)

In [ ]:
import numpy as np

thresholds = np.linspace(0.1, 0.9, 100)
f1_scores = []

for t in thresholds:
    preds = (y_pred_prob > t).astype(int)
    f1_scores.append(f1_score(y_test, preds))

best_t = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print("Optimal threshold:", best_t)
print("Best F1-score:", best_f1)